# Assignment 2: The Dirty Ledger
### ICA Week 2: Loading, cleaning and data quality

This notebook checks the supplied teaching-price CSV. I will inspect the original data first, make a separate working copy, and only make changes that I can explain. Unusual prices will be flagged rather than silently replaced.

## 1. Importing libraries

I am using pandas because it makes it straightforward to read a CSV, inspect columns and check duplicates or missing values.

In [6]:
import pandas as pd
from pathlib import Path

## 2. Loading the original CSV

Place `teaching_prices_messy.csv` in the same folder as this notebook. The original data is loaded into `raw_df`. I will not overwrite this object.

In [7]:
file_path = "C:/Users/User/OneDrive/Desktop/ICA_Fall_Project/Assignments/Week2/teaching_prices_messy.csv"
raw_df = pd.read_csv("teaching_prices_messy.csv")

work_df = raw_df.copy()


print("Original shape:", raw_df.shape)
display(raw_df.head())

Original shape: (97, 6)


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 3. Initial dataset inspection

First I check the size, column names, data types and a general summary. The price column may need attention if it has been read as text.

In [8]:
print("Rows and columns:", raw_df.shape)

print(raw_df.columns.tolist())

print("Data types:")
print(raw_df.dtypes)

print("General information:")
raw_df.info()

print("First five rows:")
display(raw_df.head())

Rows and columns: (97, 6)
['date', 'item', 'city', 'price', 'unit', 'data_label']
Data types:
date          str
item          str
city          str
price         str
unit          str
data_label    str
dtype: object
General information:
<class 'pandas.DataFrame'>
RangeIndex: 97 entries, 0 to 96
Data columns (total 6 columns):
 #   Column      Non-Null Count  Dtype
---  ------      --------------  -----
 0   date        97 non-null     str  
 1   item        97 non-null     str  
 2   city        97 non-null     str  
 3   price       96 non-null     str  
 4   unit        97 non-null     str  
 5   data_label  97 non-null     str  
dtypes: str(6)
memory usage: 4.7 KB
First five rows:


,date,item,city,price,unit,data_label
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 4. Missing-value analysis

A missing price is not the same as a price of zero. I will count missing values and keep the missing observation visible instead of guessing a replacement.

In [9]:
missing_before = raw_df.isna().sum()
print("Missing values in each column:")
display(missing_before.to_frame(name="missing_count"))

Missing values in each column:


,missing_count
date,0
item,0
city,0
price,1
unit,0
data_label,0


## 5. Duplicate analysis

I check both fully identical rows and repeated date-item-city combinations. This helps distinguish an exact repeated record from two records that may share a key but differ in other fields.

In [10]:
exact_duplicate_count = raw_df.duplicated().sum()
key_duplicate_count = raw_df.duplicated(subset=["date", "item", "city"]).sum()

print("Exact duplicate rows:", exact_duplicate_count)
print("Repeated date-item-city keys:", key_duplicate_count)

display(raw_df[raw_df.duplicated(keep=False)].sort_values(["date", "item", "city"]))

Exact duplicate rows: 1
Repeated date-item-city keys: 1


,date,item,city,price,unit,data_label
8,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
96,2026-01-08,Staple A,City 1,99.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 6. Investigating suspicious values

The price column is currently text. I create a temporary numeric version after removing commas. This is only for investigation at this stage; it does not mean the values are correct.

I also look at the item labels and the prices attached to them.

In [11]:
price_as_text = raw_df["price"].astype("string").str.replace(",", "", regex=False)
price_for_check = pd.to_numeric(price_as_text, errors="coerce")

print("Values that could not be converted (including missing values):",
      price_for_check.isna().sum())

print("Rows with a missing price:")
display(raw_df[raw_df["price"].isna()])

print("Rows with unusually high prices (screening threshold: 1000):")
display(raw_df[price_for_check > 1000])

print("Rows where item label is not exactly one of the usual labels:")
display(raw_df[~raw_df["item"].isin(["Staple A", "Oil B", "Pulse C", "Sweetener D"])])

Values that could not be converted (including missing values): 1
Rows with a missing price:


,date,item,city,price,unit,data_label
5,2026-01-01,Pulse C,City 2,NaN,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


Rows with unusually high prices (screening threshold: 1000):


,date,item,city,price,unit,data_label
17,2026-01-15,Staple A,City 2,"1,250",PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...
30,2026-01-22,Sweetener D,City 1,9999,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


Rows where item label is not exactly one of the usual labels:


,date,item,city,price,unit,data_label
40,2026-02-05,oil b,City 1,108.5,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...


## 7. Cleaning decisions

My decisions are deliberately conservative:

- Remove one fully identical duplicate from the working copy.
- Keep the original item label. Create a normalized candidate only to investigate whether `oil b` means `Oil B`.
- Check whether that possible label correction creates a duplicate date-item-city key. It does, and the two prices differ, so I do not merge the records.
- Convert the price text to a numeric column after removing commas.
- Keep the missing price as missing.
- Keep the high values (including 1250 and 9999) but flag them for review.
- Do not replace the suspicious `oil b` price. The label and the price are separate questions.

The original `raw_df` remains unchanged.

In [12]:
# Remove only the fully identical repeated row.
rows_before_cleaning = len(work_df)
work_df = work_df.drop_duplicates().copy()

# Keep the original item label. Do not merge the lowercase spelling yet.
work_df["item_original"] = work_df["item"]

# Create a possible standardized label for investigation only.
work_df["item_normalized_candidate"] = work_df["item"].replace({"oil b": "Oil B"})

# Check whether standardizing the label would create a repeated key.
normalized_key_counts = work_df.groupby(
    ["date", "item_normalized_candidate", "city"]
).size()
work_df["label_collision_flag"] = work_df.apply(
    lambda row: normalized_key_counts.loc[(
        row["date"], row["item_normalized_candidate"], row["city"]
    )] > 1,
    axis=1
)

# Convert the price to numeric. Commas are treated as thousands separators.
work_df["price"] = (
    work_df["price"]
    .astype("string")
    .str.replace(",", "", regex=False)
)
work_df["price"] = pd.to_numeric(work_df["price"], errors="coerce")

# Add flags; do not delete or alter the suspicious observations.
work_df["price_missing_flag"] = work_df["price"].isna()
work_df["price_review_flag"] = work_df["price"] > 1000

rows_after_cleaning = len(work_df)

print("Rows before:", rows_before_cleaning)
print("Rows after:", rows_after_cleaning)
print("Rows removed:", rows_before_cleaning - rows_after_cleaning)
display(work_df.head())

Rows before: 97
Rows after: 96
Rows removed: 1


,date,item,city,price,unit,data_label,item_original,item_normalized_candidate,label_collision_flag,price_missing_flag,price_review_flag
0,2026-01-01,Staple A,City 1,101.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Staple A,Staple A,False,False,False
1,2026-01-01,Staple A,City 2,107.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Staple A,Staple A,False,False,False
2,2026-01-01,Oil B,City 1,298.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Oil B,Oil B,False,False,False
3,2026-01-01,Oil B,City 2,304.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Oil B,Oil B,False,False,False
4,2026-01-01,Pulse C,City 1,200.0,PKR per kg (illustrative),ILLUSTRATIVE TEACHING DATA - NOT ACTUAL PBS OB...,Pulse C,Pulse C,False,False,False


## 8. Before-and-after comparison

This table summarizes the main checks. The aim is to show what changed, not just to show a clean-looking final table.

In [13]:
comparison = pd.DataFrame({
    "Check": [
        "Rows",
        "Columns in original dataset",
        "Missing price values",
        "Exact duplicate rows",
        "Price data type",
        "Distinct item labels"
    ],
    "Before": [
        len(raw_df),
        raw_df.shape[1],
        raw_df["price"].isna().sum(),
        raw_df.duplicated().sum(),
        str(raw_df["price"].dtype),
        raw_df["item"].nunique()
    ],
    "After": [
        len(work_df),
        len(raw_df.columns) + 5,  # two audit columns and three flags were added
        work_df["price"].isna().sum(),
        work_df.duplicated().sum(),
        str(work_df["price"].dtype),
        work_df["item"].nunique()
    ]
})
display(comparison)

,Check,Before,After
0,Rows,97,96
1,Columns in original dataset,6,11
2,Missing price values,1,1
3,Exact duplicate rows,1,0
4,Price data type,str,Float64
5,Distinct item labels,5,5


## 9. Final data-quality checks

There are 12 dates, two cities, and four likely intended item categories, so 96 combinations would be expected if the lowercase `oil b` is the same item as `Oil B`. However, testing that normalization creates a repeated key with two different prices. The notebook reports this conflict instead of choosing one price or deleting a record.

This is why the final check is not simply a pass/fail count. The data still needs clarification about that label and price.

In [14]:
date_count = work_df["date"].nunique()
raw_item_count = work_df["item"].nunique()
city_count = work_df["city"].nunique()
expected_combinations_if_four_items = date_count * 4 * city_count

# Check keys using the original labels.
original_key_counts = work_df.groupby(["date", "item", "city"]).size()
original_key_duplicates = int((original_key_counts > 1).sum())

# Also check what happens if the possible lowercase label is standardized.
normalized_key_counts = work_df.groupby(
    ["date", "item_normalized_candidate", "city"]
).size()
normalized_key_duplicates = int((normalized_key_counts > 1).sum())
observed_normalized_combinations = len(normalized_key_counts)

print("Distinct dates:", date_count)
print("Distinct raw item labels:", raw_item_count)
print("Distinct cities:", city_count)
print("Expected combinations if there are four intended items:", expected_combinations_if_four_items)
print("Original-label combinations:", len(original_key_counts))
print("Repeated original-label keys:", original_key_duplicates)
print("Normalized-label combinations:", observed_normalized_combinations)
print("Repeated keys after testing normalized labels:", normalized_key_duplicates)
print("Missing prices after cleaning:", work_df["price"].isna().sum())
print("Prices flagged for review:", work_df["price_review_flag"].sum())
print("Rows flagged for possible label collision:", work_df["label_collision_flag"].sum())

print("\nRows involved in the possible label collision:")
display(work_df[work_df["label_collision_flag"]][
    ["date", "item_original", "item_normalized_candidate", "city", "price"]
])

if normalized_key_duplicates > 0:
    print("Do not merge these labels automatically. The repeated normalized key has conflicting records.")
else:
    print("No repeated normalized keys were found.")

Distinct dates: 12
Distinct raw item labels: 5
Distinct cities: 2
Expected combinations if there are four intended items: 96
Original-label combinations: 96
Repeated original-label keys: 0
Normalized-label combinations: 95
Repeated keys after testing normalized labels: 1
Missing prices after cleaning: 1
Prices flagged for review: 2
Rows flagged for possible label collision: 2

Rows involved in the possible label collision:


,date,item_original,item_normalized_candidate,city,price
40,2026-02-05,oil b,Oil B,City 1,108.5
42,2026-02-05,Oil B,Oil B,City 1,318.0


Do not merge these labels automatically. The repeated normalized key has conflicting records.


## 10. Review flagged observations

These records remain in the working dataset. A flag means that a value deserves attention, not that it is definitely wrong.

In [15]:
display(work_df[
    work_df["price_missing_flag"] | work_df["price_review_flag"] | work_df["label_collision_flag"]
][["date", "item_original", "item", "city", "price",
   "price_missing_flag", "price_review_flag", "label_collision_flag"]])

,date,item_original,item,city,price,price_missing_flag,price_review_flag,label_collision_flag
5,2026-01-01,Pulse C,Pulse C,City 2,<NA>,True,<NA>,False
17,2026-01-15,Staple A,Staple A,City 2,1250.0,False,True,False
30,2026-01-22,Sweetener D,Sweetener D,City 1,9999.0,False,True,False
40,2026-02-05,oil b,oil b,City 1,108.5,False,False,True
42,2026-02-05,Oil B,Oil B,City 1,318.0,False,False,True


## 11. Conclusion

The file had a missing price, a repeated record, an inconsistent item label, and price values that needed investigation. I removed only the exact duplicate and converted the price column to numeric form. I did not fill the missing price or replace the unusual prices because the CSV does not provide enough evidence to know their correct values.

I initially considered standardizing `oil b` to `Oil B`, but testing that change created a repeated date-item-city key with a different price. I therefore kept the original label and flagged the possible match instead of merging the records. The working copy is more useful for analysis, but this conflict remains unresolved. The illustrative-data warning also remains important: these are teaching observations, not actual PBS data.